# 03 — Auto Merge

Optionally merges over-split units before the full analyzer is computed.

- **`enabled = false` (default):** sorting is passed through unchanged. The task
  still writes to a canonical output path so `AnalyzerTask` always has a single
  stable upstream regardless of whether merging was performed.
- **`enabled = true`:** a temporary SortingAnalyzer is built over four lightweight
  extensions (`random_spikes`, `templates`, `template_similarity`, `correlograms`),
  `sic.auto_merge_units()` is called with the configured presets, and the merged
  sorting is saved. The temporary analyzer is deleted afterwards.

Toggle merging by setting `tasks.auto_merge.enabled` in `pipeline_config.json`.
A config change invalidates the cached status, so re-running with a different
`enabled` value automatically re-processes all wells.

In [ ]:
import sys
import traceback
import logging
from pathlib import Path

import pandas as pd

_repo_root = str(Path("..").resolve())
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

from dataset_manager import DatasetManager
from pipeline_manager import PipelineManager, WorkItem
from pipeline_manager.task_record import TaskStatus
from pipeline_tasks import PreprocessingTask, SortingTask, AutoMergeTask
from config_manager import ConfigManager

logging.basicConfig(level=logging.WARNING)
print("Imports OK")

## Import auto-merge task

`AutoMergeTask` reads its upstream sorting path from
`params["sorting_output_root"]` — it does not use `data_path`.
Set `tasks.auto_merge.sorting_output_root` in `pipeline_config.json`
to match `tasks.sorting.output_root` from the sorting step.

In [ ]:
print(f"Task: {AutoMergeTask.task_name!r}  deps={AutoMergeTask.dependencies}")
AutoMergeTask.default_params()

## Config

On the **first run** this cell writes a template and stops — edit the file then re-run.

Key parameters to review:
- `global.data_root` / `global.analysis_root`
- `tasks.auto_merge.sorting_output_root` — must match `tasks.sorting.output_root`
- `tasks.auto_merge.enabled` — set `true` to activate merging
- `tasks.auto_merge.presets` — list of auto-merge preset names

In [ ]:
CONFIG_FILE = Path("../pipeline_config.json")

cm = ConfigManager()
cm.set_global("data_root", "/path/to/raw/data")
cm.set_global("analysis_root", "/path/to/analysis")
cm.register_task(AutoMergeTask)

if not CONFIG_FILE.exists():
    cm.generate_template(CONFIG_FILE)
    raise RuntimeError(
        f"Template written to {CONFIG_FILE}.\n"
        "Edit it, then re-run this cell."
    )

cm.load(CONFIG_FILE)

DATA_ROOT     = Path(cm.get_global("data_root"))
ANALYSIS_ROOT = Path(cm.get_global("analysis_root"))

print(f"Config loaded from: {CONFIG_FILE}")
print(f"  data_root:     {DATA_ROOT}")
print(f"  analysis_root: {ANALYSIS_ROOT}")
print(f"  task params:   {cm.get_task_params(AutoMergeTask.task_name)}")


## 1. Scan recordings

In [ ]:
ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)

dataset_mgr = DatasetManager(DATA_ROOT, ANALYSIS_ROOT)
recordings  = dataset_mgr.get_recording_by([("scan_type", "==", "Network")])

print(f"Found {len(recordings)} Network recording(s)")

summary_rows = [
    {
        "recording_key": r.cache_key,
        "date":          r.date,
        "sample_id":     r.sample_id,
        "plate_id":      r.plate_id,
        "n_wells":       len(r.wells),
    }
    for r in recordings
]
pd.DataFrame(summary_rows)


## 2. Register task and add wells

In [ ]:
TASK_NAME = AutoMergeTask.task_name
REGISTERED_TASK_CLASSES = [PreprocessingTask, SortingTask, AutoMergeTask]

pipeline_mgr = PipelineManager(ANALYSIS_ROOT, config_provider=cm)

for cls in REGISTERED_TASK_CLASSES:
    try:
        pipeline_mgr.register_task(cls)
        print(f"Registered task: {cls.task_name!r}")
    except ValueError as e:
        print(f"Task already registered ({e}) — continuing")

n_added = 0
n_skipped = 0
for rec in recordings:
    if not rec.h5_recordings:
        print(f"WARNING: no h5 structure for {rec.cache_key!r} — skipping")
        n_skipped += 1
        continue
    for rec_name, well_ids in rec.h5_recordings.items():
        for well_id in well_ids:
            pipeline_mgr.add_well(rec.cache_key, f"{rec_name}/{well_id}")
            n_added += 1

print(f"\nWork units registered: {n_added}  |  Skipped: {n_skipped}")


## 3. Pipeline status overview

In [ ]:
def _status_table(mgr: PipelineManager, task_name: str) -> pd.DataFrame:
    rows = []
    for entry in mgr.entries:
        if "/" not in entry.well_id:
            continue
        rec_name, well_id = entry.well_id.split("/", 1)
        task = entry.tasks.get(task_name)
        rows.append({
            "recording_key": entry.recording_key,
            "rec_name":      rec_name,
            "well_id":       well_id,
            "status":        task.status if task else "—",
            "config_fresh":  mgr.is_task_complete(WorkItem(entry.recording_key, entry.well_id, task_name))
                             if task and task.status == TaskStatus.COMPLETE else "—",
            "output_path":   str(task.output_path) if task and task.output_path else "",
            "error":         (task.error or "")[:120] if task else "",
        })
    return pd.DataFrame(rows)


df_status = _status_table(pipeline_mgr, TASK_NAME)
print(df_status["status"].value_counts().to_string())
df_status


## 4. Run auto-merge

When `enabled = false`, each well completes in seconds (load + save pass-through).
When `enabled = true`, expect a few minutes per well depending on unit count and
the chosen merge presets.

To retry failed wells without resetting all: set `RETRY_FAILED = True`.  
To reset a specific well: `pipeline_mgr.refresh(TASK_NAME, recording_key=..., well_id="rec0000/well000")`  
To reset everything: `pipeline_mgr.refresh(TASK_NAME)`

In [ ]:
RETRY_FAILED = False

task   = AutoMergeTask()
params = cm.get_task_params(TASK_NAME)

_rec_lookup = {r.cache_key: r for r in recordings}
_current_recording_keys = set(_rec_lookup.keys())

while True:
    batch_size = max(1, len(pipeline_mgr.entries) * max(1, len(REGISTERED_TASK_CLASSES)))
    work_items = [
        item
        for item in pipeline_mgr.get_next_task(
            n=batch_size,
            retry_failed=RETRY_FAILED,
            recording_keys=_current_recording_keys,
        )
        if item.task_name == TASK_NAME
    ]
    if not work_items:
        break

    item      = work_items[0]
    rec_entry = _rec_lookup[item.recording_key]
    rec_name, well_id = item.well_id.split("/", 1)

    print(f"\nProcessing: {item.recording_key} / {rec_name} / {well_id}")
    pipeline_mgr.update_status(item, TaskStatus.RUNNING)

    try:
        output_path = task.run(
            item.recording_key,
            item.well_id,
            dataset_mgr.get_path(rec_entry),
            params,
        )
        pipeline_mgr.update_status(item, TaskStatus.COMPLETE, output_path=output_path)
        print(f"  ✓  saved → {output_path}")
    except Exception as exc:
        pipeline_mgr.update_status(item, TaskStatus.FAILED, error=str(exc))
        traceback.print_exc()
        print(f"  ✗  failed: {exc}")

print("\nDone — no ready target tasks remain.")


## 5. Final status report

In [ ]:
df_final = _status_table(pipeline_mgr, TASK_NAME)

print("=== Summary ===")
print(df_final["status"].value_counts().to_string())

stale = df_final[
    (df_final["status"] == TaskStatus.COMPLETE) & (df_final["config_fresh"] == False)
]
if not stale.empty:
    print(f"\n\u26a0\ufe0f  {len(stale)} well(s) completed with outdated config.")

failed = df_final[df_final["status"] == TaskStatus.FAILED]
if not failed.empty:
    print("\n=== Failed entries ===")
    for _, row in failed.iterrows():
        print(f"  {row['recording_key']} / {row['rec_name']} / {row['well_id']}")
        print(f"    error: {row['error']}")

df_final